In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import csv

start_date = '2025-01-01'
end_date   = '2025-11-15'
keyword = input('찾고자 하는 주식회사 명을 입력하시오: ').strip()

def daterange(s, e):
    start = datetime.strptime(s, "%Y-%m-%d").date()
    end = datetime.strptime(e, "%Y-%m-%d").date()
    while start <= end:
        yield start.strftime("%Y-%m-%d")
        start += timedelta(days=1)

rows = []

for date in daterange(start_date, end_date):
  print(date)
  page_number = 1
  prev_titles = set() #중복 데이터 저장

  while True:
      url = f'https://finance.naver.com/news/mainnews.naver?date={date}&page={page_number}'
      response = requests.get(url)
      soup = BeautifulSoup(response.text, 'html.parser')

      # 크롤링하는 날의 탈출 조건-1: 기사가 없을 때
      titles = soup.select("dd.articleSubject > a")
      if not titles:
          break
      # 크롤링하는 날의 탈출 조건-2: 이전 페이지와 동일(마지막 페이지 반복)일 때
      current_titles = {t.get_text(strip=True) for t in titles}
      if current_titles == prev_titles:
          break
      prev_titles = current_titles

      # 제목에서 키워드가 있는 것만 rows에 date, title 입력
      for t in titles:
          text = t.get_text(strip=True)
          if keyword in text:
              print(text)
              rows.append({
                  "date": date,
                  "title": text,
              })

      page_number += 1

#csv파일에다가 date,title로 저장
outfile = f"/content/drive/MyDrive/Colab Notebooks/Bamboo DnA(25.9~25.12)/2024_stock_data/naver_finance_titles_{keyword}_{start_date}_to_{end_date}.csv"
with open(outfile, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["date", "title"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\n저장 완료: {outfile} (총 {len(rows)}건)")


찾고자 하는 주식회사 명을 입력하시오: LG
2025-01-01
1월 IPO 기업 9곳…'6조 몸값' LG CNS 출격대기
2025-01-02
공모주 한파 이겨낼까 … LG CNS 등 9곳 출격
'6조 대어' LG CNS, 얼어붙은 공모주 시장 녹일까
LG이노텍, 4분기 실적 부진 전망에 새해 첫 거래일부터 하락
메리츠證 "LG이노텍, 고객사 내 경쟁 심화 우려…목표가↓"
LG이노텍, 4Q 영업익 시장 예상치 하회 전망…목표가↓ -메리츠
2025-01-03
삼성SDS 주가 하락에…상장 앞둔 LG CNS 고심
2025-01-04
2025-01-05
혹한기 겪은 IPO 시장, LG CNS로 `봄` 맞을까
업력 39년 'LG CNS'…왜 지금 IPO할까[공모주달력]
2025-01-06
LS증권, 'LG·우투증권 DNA' 강화
LG전자, 목표주가 14만→12.5만-하나
2025-01-07
2025-01-08
'어닝쇼크' 삼성전자·LG전자 반등… 종목장세 온다
삼전·LG '온기'로 시작한 4Q 어닝 시즌…코스피 기지개 켤까
LG전자 작년 매출 87.7조 '역대 최대'…4분기는 예상보다 부진(종합)
2025-01-09
LG CNS "해외기업 인수추진…곧 깜짝발표"
‘6조 대어’ LG CNS 출격…꽁꽁 얼어붙은 공모주 시장 녹이나
'6조 대어' LG CNS “DX 기업 인수 등에 3300억 투자…깜짝 뉴스 나올 것” [시그널]
‘최대어’ LG CNS 수요 예측 ‘주목’…IPO 시장, 분위기 반전 ‘이목’
길어지는 '이차전기 혹한기'…LG에너지솔루션 4% '뚝'[핫종목]
'6조 대어' LG CNS "AX 선도 기업 자리매김 할 것···중복 상장 아냐"
증시 한파 속 IPO 나선 LG CNS…"투자재원 6000억 확보"
LG에너지솔루션 적자 전환 '실적 쇼크'에 4%대 급락
LG CNS “대주주는 LG지만 중복상장 절대 아냐…오히려 LG 주주에 이득"
‘올해 첫 코스피 상장’ LG CNS “韓경제에 긍정적 신호 만들겠다”
“글로벌 DX기업으로”…LG CNS, IPO 침체 파고

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Bamboo DnA(25.9~25.12)/2024_stock_data/naver_finance_titles_LG_2025-01-01_to_2025-11-15.csv'

In [ ]:
#LG에너지솔루션=LG에너지솔루션 클롤링+LG크롤링후 해당되는 단어만 필터링
import pandas as pd

file1 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Bamboo DnA(25.9~25.12)/2024_stock_data/naver_finance_titles_LG에너지솔루션_2024-01-01_to_2024-12-31.csv", encoding='utf-8')
file2 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Bamboo DnA(25.9~25.12)/2024_stock_data/naver_finance_titles_LG_2024-01-01_to_2024-12-31.csv", encoding='utf-8')

merged = pd.concat([file1, file2], ignore_index=True)
merged = merged.drop_duplicates(subset=["date", "title"])
merged = merged.sort_values(by="date").reset_index(drop=True)

mask = merged["title"].str.contains("LG에너지솔루션|LG엔솔", case=False, na=False)
filtered = merged[mask].reset_index(drop=True)

filtered.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Bamboo DnA(25.9~25.12)/2025_stock_data/naver_finance_titles_LG에너지솔루션_LG엔솔_2024.csv",
    index=False, encoding="utf-8-sig"
)


In [ ]:
import pandas as pd
import re
from io import StringIO

# 1. 사용자 데이터 (여기에 데이터를 계속 추가해도 됩니다)
raw_text = """
2025-01-01
1월 IPO 기업 9곳…'6조 몸값' LG CNS 출격대기
2025-01-02
공모주 한파 이겨낼까 … LG CNS 등 9곳 출격
'6조 대어' LG CNS, 얼어붙은 공모주 시장 녹일까
LG이노텍, 4분기 실적 부진 전망에 새해 첫 거래일부터 하락
메리츠證 "LG이노텍, 고객사 내 경쟁 심화 우려…목표가↓"
LG이노텍, 4Q 영업익 시장 예상치 하회 전망…목표가↓ -메리츠
2025-01-03
삼성SDS 주가 하락에…상장 앞둔 LG CNS 고심
2025-01-04
2025-01-05
혹한기 겪은 IPO 시장, LG CNS로 `봄` 맞을까
업력 39년 'LG CNS'…왜 지금 IPO할까[공모주달력]
2025-01-06
LS증권, 'LG·우투증권 DNA' 강화
LG전자, 목표주가 14만→12.5만-하나
2025-01-07
2025-01-08
'어닝쇼크' 삼성전자·LG전자 반등… 종목장세 온다
삼전·LG '온기'로 시작한 4Q 어닝 시즌…코스피 기지개 켤까
LG전자 작년 매출 87.7조 '역대 최대'…4분기는 예상보다 부진(종합)
2025-01-09
LG CNS "해외기업 인수추진…곧 깜짝발표"
‘6조 대어’ LG CNS 출격…꽁꽁 얼어붙은 공모주 시장 녹이나
'6조 대어' LG CNS “DX 기업 인수 등에 3300억 투자…깜짝 뉴스 나올 것” [시그널]
‘최대어’ LG CNS 수요 예측 ‘주목’…IPO 시장, 분위기 반전 ‘이목’
길어지는 '이차전기 혹한기'…LG에너지솔루션 4% '뚝'[핫종목]
'6조 대어' LG CNS "AX 선도 기업 자리매김 할 것···중복 상장 아냐"
증시 한파 속 IPO 나선 LG CNS…"투자재원 6000억 확보"
LG에너지솔루션 적자 전환 '실적 쇼크'에 4%대 급락
LG CNS “대주주는 LG지만 중복상장 절대 아냐…오히려 LG 주주에 이득"
‘올해 첫 코스피 상장’ LG CNS “韓경제에 긍정적 신호 만들겠다”
“글로벌 DX기업으로”…LG CNS, IPO 침체 파고 넘을까
‘6조 대어’ 꿈 이룰까…LG CNS 기관 수요예측 돌입, 확약률이 관건 [투자360]
2025-01-10
회사채 흥행…삼성증권 ‘2조’, LG유플러스 ‘3조’ 모아[마켓인]
LG유플러스, 회사채 수요예측서 3조500억 모여
NICE신평, LG화학 등급전망 ‘부정적’…LG엔솔 실적 연동[마켓인]
올해 CES 인기 검색어는 '엔비디아'…삼성·LG는 4·5위
LG·롯데·두산 계열사 상반기 IPO 신고식‥K증시, 분위기 반전할까
LG엔솔 '어닝쇼크'…"일회성 비용 여파, 테슬라 신차 수혜 기대"
올 첫 IPO 대어 LG CNS “DX 선도 시장 목표…곧 깜짝 뉴스 나올 것” [시그널]
'5조 몸값' 케이뱅크도 미뤘는데…'IPO 대어' LG CNS가 공모주 시장 녹일까
LG전자, 어닝 쇼크에 훅 낮아진 눈높이
2025-01-11
업황 악화에 LG엔솔도 부진…위기의 LG화학[위클리 크레딧]
LG CNS, 빠른 신고서 약됐다?…피어그룹 주가 날고 맥쿼리PE 엑시트 ‘기대’[투자360]
2025-01-12
'6조 몸값' LG CNS 온다… '공모주 월척' 잡아볼까
'LG엔솔' 이후 최대어 온다...연초 IPO 신규 상장 훈풍
연초효과에 회사채 슈퍼위크…‘SK하이닉스·LG화학’ 조달 시동
'6조 대어' LG CNS는 수요예측 흥행 성공할까[공모주달력]
2025-01-13
美 보조금 줄어들라…LG엔솔 눈높이 '뚝'
거버넌스포럼 "LG CNS, 사실상 중복상장…지주사 부작용 최소화해야"
LG에너지솔루션, 실적 부진에 증권가 목표가 줄줄이 '하향'
신한證 "LG디스플레이, 올해 실적 기대감은 여전..목표가 ↓"
"배터리 부문 적자 전환" LG화학, 목표가↓ - IBK
IBK證 "LG화학, 배터리·소진 시차 고려해야…목표가 20%↓"
"LG화학, 모든 사업부문 부진에 적자전환 전망돼…목표가↓"-IBK
LG화학, 4분기 적자전환 전망…목표가↓-IBK
2025-01-14
'종합IT부품사' 진화 목표 LG이노텍…순매수 1위[주식 초고수는 지금]
'테슬라 호재'에 2차전지株도 상승세…LG엔솔 4거래일만 반등[핫종목]
LG CNS 상장 앞두고…LG 주가 1년 전으로 '회귀' [종목+]
금주 5개사 공모주 청약…'6조 대어' LG CNS 'IPO 한파' 녹일까
IPO 시장 칼바람인데…전문가 57% "LG CNS 공모가 비싸다"
2025-01-15
LG CNS, 수요예측 흥행 기대감…“기관 분위기 좋다” [시그널]
2025-01-16
작년 5조 모았던 LG엔솔 회사채, 1년 만에 분위기 반전[마켓인]
LG CNS 수요예측 흥행에 ㈜LG도 웃는다
2025-01-17
LG CNS, 공모가 최상단 6만1900원…수요예측 76조 몰려
'6조 대어' LG CNS, 공모가 6만1900원 확정
LG CNS, 공모가 상단 확정… 시가총액 6조원 정조준
'IPO 대어' LG CNS 공모가 6만1천900원…수요예측 경쟁률 114대1
LG CNS, 공모가 6만1900원 확정 '밴드 상단'
LG화학, 회사채 수요예측서 1.6조 모아…아쉬운 금리[마켓인]
LG CNS '몸값 6조' …LG엔솔 이후 최대규모 [시그널]
LG화학, 신용등급 전망 하향에도 회사채 수요 탄탄…모집액 6배 모아
인벤테라, LG화학과 차세대 MRI 조영제 신약 원료 생산 위수탁계약
인벤테라, LG화학과 차세대 MRI 조영제 신약 원료 생산 위수탁계약
SK하이닉스·LG에너지 나홀로 상승… 코스피 2510선 '강보합권'
"LG, 밸류업 통한 재평가 기대" 목표가 11만→9.5만 하향-흥국
2025-01-18
3년만의 최대어 잡아볼까…LG CNS발 공모주 열풍 재점화
2025-01-19
'6조 대어' LG CNS, 21일부터 청약…'엔솔' 열풍 재현할까
‘6兆 대어’ LG CNS, 21일부터 청약
몸값 '6兆' LG CNS, 이번주 청약 개시…흥행 열기 이어갈까
'3년 만에 최대어' LG CNS 시총 6조 확정…IPO 청신호에 시장 '활기'
'겸손한 몸값' 강조한 LG CNS' 청약…기관 확약은 저조[공모주달력]
트럼프 ‘입’에 다시 주목… SK하닉·LG엔솔 실적 발표 릴레이[주간증시전망]
2025-01-20
LG생건 어닝쇼크에도 ··· 화장품주 기대감 솔솔
LG화학 ‘부정적’ 아웃룩…LG엔솔은 ‘안정적’인 이유[마켓인]
LG에너지솔루션, 테슬라 급등·저가 매수세 유입에 2% 상승[핫종목]
이번주 공모주 청약, LG CNS 등 3개사
2025-01-21
'공모주 대어' LG CNS, 청약 첫날 증거금 1조원 모아…"내일 눈치싸움 치열"
[MK 골든크로스 돌파종목 : LG에너지솔루션(373220) & 써니전자(004770)]
'6조 대어' LG CNS, 일반청약 시작…공모주 시장 살릴까
2025-01-22
LG CNS 일반청약에 증거금 21조원 몰려…IPO 시장 온기들까
‘상반기 IPO 대어’ LG CNS 일반 청약에 21兆 몰려
LG CNS에 21조 뭉칫돈…경쟁률 123대1 [시그널]
LG CNS 공모 청약 흥행 … 21조 몰렸다
체질 개선 나선 LGD, 1년만 분기 흑자…OLED 매출 비중 최대(종합2보)
체질 개선 나선 LGD, 1년만에 분기 흑자…OLED 매출 비중 최대(종합)
'6조 대어' LG CNS, 우리사주조합 청약 '선방'…청약률 82%
2025-01-23
LG CNS, 3주 받고 상장 당일 팔까…'따블' 매도 시 수익은?
LG이노텍, 지난해 실적 부진에 6%↓…52주 신저가[핫종목]
코스피 첫 대어, LG CNS 21조 몰렸다…IPO 시장 훈풍 부나 [투자360]
‘21조’ 몰린 LG CNS, 공모 청약 흥행...IPO 시장 ‘훈풍’ 기대
2월 MSCI 편출 종목?…롯데케미칼·포스코DX·LG화학우 등 대거 전망 [투자360]
메리츠證 "LG이노텍, WWDC 전후 주가 반전 기대"
LG디스플레이, 현재 주가 매력적…목표가는↓ -SK
올 '최대어' LG CNS 공모주 청약 흥행 [시그널]
2025-01-24
LG엔솔, 작년 영업익 73% 급감…"현지화·운영 효율화 총력"(종합2보)
美 로봇사 인수 LG전자, 인도 IPO 기대감…순매수 1위[주식 초고수는 지금]
LG전자, 美 베어로보틱스 인수...너도 나도 '미래 먹거리는 로봇' [오한마]
2025-01-25
LG CNS로 6000억 번 맥쿼리PE, 롯데글로벌로지스 FI는 어떨까 [투자360]
2025-01-26
2025-01-27
로봇 기업 품는 삼성·LG…개발 경쟁 '활활'
근원약 없는 글로벌 통풍치료시장...3상 진입 LG화학,중외제약 전망은
2025-01-28
LG CNS 흥행에…설 연휴 이후 IPO '대어' 입성 빨라질까
2025-01-29
2025-01-30
LG화학 찜 재영텍, 세마인베 투자유치[fn마켓워치]
2025-01-31
"LG에너지솔루션, 올해 낙관적 가이던스..목표가 55만→40만" -DB금투
“LG, ‘6조 대어’ LG CNS 상장 영향 크지 않을 것”…목표가↓
"LG, 자회사 주가·실적↓…목표가 하향"[클릭 e종목]
LG엔솔, 매출 반등 전망되나 실적·가격 부담 여전…목표가↓-하나
2025-02-01
IPO `대어` LG CNS 다음주 코스피 입성… 주가는 어떻게?
올해 첫 IPO ‘大漁’ 뜬다…LG CNS 내주 코스피 입성
올해 첫 '대어' LG CNS 다음주 코스피 입성[IPO챗]
2025-02-02
연휴 직후 '대어' 온다… LG CNS 다음주 코스피 입성
설 연휴 끝나자 회사채 우르르…대어 LG엔솔, 최대 1.6조 조달
LG CNS, 연휴 직후 코스피 입성…상반기 IPO 활기 불어 넣는다
LG CNS 코스피 상장, 더본코리아 락업 해제[공모주달력]
2025-02-03
IPO 시장 옥석가리기 본격화…LG CNS ‘긴장’
“따따블 가능해요?”…21兆 몰린 LG CNS, 상장 초읽기
연일 신저가 LG화학, ELS 손실 어쩌나
'IPO 순항' LG CNS, 상장일 성적에 쏠리는 눈
관세전쟁에 급락…지금이 매수기회? SK하닉·LG이노텍 등 순매수[주식 초고수는 지금]
21조 몰린 '대어' LG CNS, 5일 코스피 입성…"IPO 시장 훈풍 기대"
2025-02-04
새내기주 잔혹사 현재진행형…‘내일 상장’ LG CNS 시험대
LG CNS IPO 흥행 여부에 엔무브 상장 앞둔 SK그룹도 ‘촉각’
iM證 “LG화학, 전기차도 석유화학도 어렵다… 목표가 47만→32만”
“‘양극재’에 발목 잡힌 LG화학, 올해도 실적 부진 이어갈듯”…목표가↓
2025-02-05
“모처럼 대어에 ‘따블’ 기대했건만”…공모주 한파에 고개 숙인 LG CNS
혹독한 '공모주 한파' LG CNS도 못피했다
LG CNS, IPO 한파 면치 못했다···상장일 공모가 밑으로 추락
“청약 투자자 모두 손실”…LG씨엔에스마저 상장 흑역사
LG CNS, 상장 첫날 '호된 신고식'
또 터진 공모가 거품… LG CNS 10% ↓[기획]
“최대어도 한파 못 피했다” LG CNS, 코스피 입성 첫날 10% ‘뚝’ [투자360]
상장 첫날 10% 빠진 LG씨엔에스…"공모가 비쌌다" 터져나온 한탄
"대어급도 쉽지 않네" LG씨엔에스, 첫날 성적표 파란불… IPO 시장 긴장
LG CNS도 못 녹인 공모주 한파 … 공모가 밑돌아
“반년치 공모주 이익 날렸다”… LG CNS 청약 개미, 상장 첫날 평가손 340억원
LG CNS... 상장 첫날 혹독한 신고식
공모주 한파 '대어' LG CNS도 덮쳤다…첫날 공모가 밑돌아
IPO 대어 LG씨엔에스…상장 첫날 공모가 밑돌며 거래 마쳐
“본전도 못 찾았다”…LG CNS 혹독한 신고식에 개미들 ‘울상’
외국인·기관 매수에 이틀째 상승...LG CNS 9%대 ↓[코스피 마감]
계속되는 IPO 찬바람…LG CNS, 코스피 입성 첫날 10% 급락(종합)
LG CNS, 호된 코스피 신고식… 장중 5.5% 하락
'6조 대어'인데… LG씨엔에스, 상장 첫날 공모가 대비 3% '뚝'
LG씨엔에스, 장 초반 공모가 밑돌아…6%대 약세[핫스탁]
'공모주 한파' 공모가 밑도는 새내기주… LG CNS로 기지개 켤까
“시간 더 필요” vs “더 나빠지기 어렵다”…엇갈리는 LG생활건강 전망
"대어 나가신다" LG CNS, 코스피 신고식… 유통물량·증시는 '변수'
2025-02-06
123억 세금 소송 패소 ‘LG 맏사위 윤관’…수천 억 세금 맞을 수도
“LG CNS 너마저”… IPO 한파에 올해 8개 중 7개 새내기株 ‘첫날 마이너스’
LG엔솔, 회사채 수요예측서 3.7조 모아…악재 딛고 흥행[마켓인]
LG엔솔, 회사채 수요예측서 3조7450억 몰려
'호된 신고식' 치른 LG CNS, 상장 이틀째 소폭 반등
최대어 LG CNS, 충격의 상장 첫날…공모주 한파 지속되나
올해 최대어 LG CNS, 상장 첫날 '호된 신고식'
2025-02-07
1년새 '82만→116만원' 역대 최고가…삼바, LG엔솔 제쳤다
시총 80조 뚫은 삼바, LG엔솔 제치고 3위 올랐다
“전자 우려에도 바이오는 잘 나가네”...삼바, LG엔솔 제치고 시총 3위
삼성바이오 6.5% 올라 역대 최고가…LG엔솔 제치고 '시총 3위'(종합)
'상장 첫날 급락' LG CNS "저가 매수 기회"…순매수 1위[주식 초고수는 지금]
대신證 “LG, 떨어지는 자회사 가치… 목표가 9.5만→8.3만”
2025-02-08
“맥쿼리 지분 21% 잠재 매물인데…” 시장 한파에 공모가 방어 실패 LG CNS ‘고심’ [투자360]
2025-02-09
올해 첫 코스피 상장도 안통해...꽁꽁언 투심에 LG 목표가↓
LG CNS ‘비싼 상장’은 누구 배만 불리나[위클리 IB]
2025-02-10
"믿을 구석은 해외 사업인데" LG생활건강, 목표가↓ -유안타
"LG전자, 주가 과한 우려 선반영…펀더멘털·모멘텀 개선 전망"-메리츠
2025-02-11
‘실적 회초리’ 먼저 맞은 LGU+ 반등…SKT·KT는 어닝쇼크 공개 전 1위 쟁탈전
2025-02-12
SK·한화·LG만 7조 찍었다…연초 회사채 '문전성시'[마켓인]
2025-02-13
LG엔솔이 선택한 대진첨단소재 "ESS로 사업 다각화"
2차전지주, 자동차 상호관세 제외 기대감에 상승… LG엔솔 3% 올라
2025-02-14
2025-02-15
2025-02-16
고점 대비 반토막 난 LG엔솔, 여전히 ‘비싸다’는 이유…PER은 이렇게도 씁니다 [투자뉴스 뒤풀이]
서울보증보험 수요예측?LG CNS 락업해제[공모주달력]
2025-02-17
2025-02-18
개인 대상 IR 나선 애드바이오텍 “LG전자 이어 대형 렌탈사 고객사 확보... 상반기 대규모 발주”
2025-02-19
LG에너지솔루션, 도요타 러브콜에 4%↑…2차전지 株 동반 상승
2025-02-20
‘한한령 해제 기대감’에 K엔터·뷰티 강세…YG엔터·LG생건·아모레 5% 넘게 급등 중
2025-02-21
현대차證, “LG디스플레이, 올해 실적 개선세 이어갈 것… 무난한 흑자 기대”
"보름 만에 17% 뚝"…LG CNS 공모주 개미들 눈물
2025-02-22
믿었던 LG CNS 마저…주가 하락에 개미투자자만 ‘눈물’
2025-02-23
2025-02-24
2차전지株 긴 터널 끝났나 LG엔솔·삼성SDI 반등세
LG CNS, 상장 후 주가 저평가… 회복 기대감[체크!코스피]
KB證, “LG전자 가전 사업서 사상 최대 실적 전망… 목표가 9% 상향”
“LG전자, 올해 가전사업 사상 최대 실적 전망”…목표가↑
LG전자, 올해 가전 사업 실적 사상 최고치 전망…목표가↑-KB
"LG전자, 올해 가전 사업서 최대 실적 전망…목표가↑"-KB
2025-02-25
"땡큐 테슬라" 성우, LG엔솔 덕분에 '상한가'[체크!코스닥]
2025-02-26
2025-02-27
"악재보다 호재에 민감한 시기" LG이노텍 목표가 상향 -iM증권
iM증권 "LG이노텍 목표가↑…애플 관련 긍정 이벤트 많아"
iM證 “LG이노텍, 주가 싸고 애플 이벤트 이어져… 목표가 상향”
LG이노텍, 상반기 주가 반등 기대…애플 관련 이벤트도 호재 -iM
"LG이노텍, 6월까지 긍정적 주가 흐름 전망…목표가↑"-iM
LG전자, 사상 최대 실적 전망…트럼프 시대가 '기회'-KB
2025-02-28
LG이노텍, 신저가 찍고 반등 성공…주가 회복 이어질까
2025-03-01
2025-03-02
2025-03-03
2025-03-04
2025-03-05
'수익률 -25%' LG CNS, 상장 한 달째…성장 동력 없나
2025-03-06
LG이노텍 1분기 깜짝실적 전망에 주가 9% 넘게 급등(종합)
11% 배당에도…서울보증 '제2의 LG CNS' 되나  [마켓딥다이브]
‘어닝 서프라이즈’ 기대감에 LG이노텍 주가 7% 넘게 상승 [줍줍리포트]
대신증권 "LG이노텍 1분기 깜짝 실적 기대…목표가 4%↑"
“LG이노텍, 올해 1분기 아이폰 흥행 등 깜짝실적 예상”…목표가↑
LG이노텍, 1Q 깜짝실적 예상…목표가↑-대신
"LG이노텍, '깜짝 실적' 기대에도 역사적 저평가…목표가↑"-대신
2025-03-07
CEO 직접 PT에 삼성證, 범LG 문 처음으로 열었다[fn마켓워치]
LG이노텍, 단기 트레이딩 기회 부각...구조적 우려는 지속-메리츠
2025-03-08
2025-03-09
2025-03-10
LG그룹사 시총 3위 지켜낼까…상장사 주가 주목
LG디플 유증 참여한 임직원, 재미 못 봤다… 사측은 우리사주 이자 지원 연장
필에너지, ‘레이저 노칭’ 테스트 끝... “삼성·LG엔솔·유럽車업체 등에 샘플 제공 예정”
2025-03-11
테슬라 폭락 '불똥'… LG엔솔·포스코홀딩스 급락
테슬라 폭락하자 이차전지株 줄하락…LG엔솔 -2.4%[투자360]
테슬라 폭락에 이차전지株 '우수수'…LG엔솔 2.4%↓(종합)
2025-03-12
LG전자, MS와 AI·양자컴퓨팅 협력 강화 기대-KB
2025-03-13
엠플러스, LG엔솔 전고체 장비 공급 소식에 장중 급등
2025-03-14
DB손보, HD현대, LG생건… 다음주 넥스트레이드 거래 종목 100개 추가
"양자컴퓨터 급물살…주도주는 LG전자·삼성전자·SKT"-KB
KB증권 "양자컴퓨터 현실화…삼성전자·LG전자·SK텔레콤 수혜"
“양자컴퓨터 개발 급물살…LG전자·삼전·SKT 최대 수혜”
2025-03-15
2025-03-16
DB손보·LG생건·이마트도 퇴근 후 거래한다
2025-03-17
작년 LG전자 연봉 1위는 조주완…직원 평균연봉 1억1천700만원
LG전자, 양자컴과 동맹…34조 시장 수혜 대기 [마켓딥다이브]
대체거래소 거래종목 110개로 확대…신세계·LG생건 등 합류
대체거래소 거래종목 110개로 확대…이마트·LG생활건강 등 추가
DB손보·LG생건·HD현대도 출퇴근길 거래…대체거래소 거래종목 100개 늘어난다 [투자360]
"LG전자, 4년만에 최대 실적 전망…목표가↑"[클릭 e종목]
LG전자, 4년만 최대실적 전망…목표주가 13만원 ↑-KB
2025-03-18
2025-03-19
"긴 터널의 끝 보인다"…속 끓던 'LG 모태' 올해는 웃을까 [종목+]
2025-03-20
2025-03-21
국민연금, LG화학 신학철·셀트리온 서정진 사내이사 선임에 '반대'
2025-03-22
2025-03-23
2025-03-24
2025-03-25
LG에너지솔루션, 수천억원대 폴란드 계약에 3% ‘쑥’…유럽 ESS 배터리 현지생산 확대 [종목Pick]
“레뷰코퍼레이션, 지분 매각 본격화…LG유플러스 인수 가능성”[클릭 e종목]
2025-03-26
코스피, 2640선 회복…LG엔솔 '불기둥' [박해린의 마켓톡톡]
코스피, 외인·기관 매수에 2640선 회복…LG엔솔 7%↑
"LG회장 한 마디, 이 정도였어?"…LG에너지솔루션 8% 급등 [줍줍리포트]
2025-03-27
“배당락 주의하세요”…삼성전자·LG전자 등 131개 종목 31일 배당기준일
LG엔솔, 회장님 한마디에 '들썩'…반등세 이어갈까 [종목+]
2025-03-28
LG 주가 하락에… 구본준 LX 회장, 아들에게 또 ‘증여 취소 후 재증여’
2025-03-29
구광모 '밀어주기' 효과?…이번 주 8% 뛴 LG엔솔[종목현미경]
2025-03-30
삼성전자·LG엔솔·에코프로, 대차잔고 급증… 공매도 타깃 되나?
LG CNS 상장 주관…KB證 주식·채권발행 1위 석권
2025-03-31
LG전자, 러시아 돌아가나…'1호 복귀 기업'에 쏠리는 눈
코스피, 美상호관세·공매도 재개에 외국인 매도 폭탄…LG엔솔 6%·에코프로 12%↓
‘LG CNS 대표주관’ KB증권, 1분기 IPO 리그테이블 1위
공매도 재개, 삼성전자·LG엔솔 실적 발표[주간경제전망]
2025-04-01
LG엔솔, 美 얼티엄셀즈 3공장 인수…"리밸런싱으로 캐즘 극복"(종합)
2025-04-02
KB증권, LG CNS로 날았다…1Q IPO 실적 `톱`
2025-04-03
2025-04-04
2025-04-05
범현대가 목표가 상향 vs 범LG가 목표가 하향[청기홍기]
2025-04-06
2025-04-07
LG엔솔 '깜짝 실적'에도 캐즘 극복은 난망…美 생산으로 승부
LG엔솔 1분기 영업익 3천747억원…AMPC 효과에 '깜짝 실적'(종합)
“LG전자, ‘관세 대응 능력’ 증명에 주가 반등 달렸다”
LG전자, 2분기 실적이 분수령…“관세 대응 증명”-한국
2025-04-08
“관세 불확실성 남아있다”…LG전자 목표가 줄하향에 속타는 개미
LG엔솔 실적에 엇갈린 시선…"불확실성 여전" vs "최선호주"
2025-04-09
나신평 “美 관세, 국내 수출산업 저하 전망···2차전지 부담에 LG·SK·포스코 실적 둔화 우려” [투자360]
2025-04-10
LG향 공급 재개에 SK IET 주가 10%대 강세
"이런 날 왜 혼자 못 오르니"…'경기방어株' SKT·LG유플러스, 약세 [이런국장 저런주식]
2025-04-11
삼성·LG전자,  1분기 실적은 뒷전? …트럼프 한 마디에 휘둘리는 주가
한투·한국타이어, 고려아연 출자 해소…LG화학은 상호출자 유지
2025-04-12
2025-04-13
2025-04-14
코스피, 상승 마감… LG엔솔·삼성전자 등 대형주가 끌어올려
힘 잃은 트럼프의 엄포…'아이폰 테마' LG이노텍·비에이치 급등
2025-04-15
2025-04-16
'캐즘·관세 이중고' 이차전지, 업종 내 희비…LG엔솔만 '활짝'
올해 실적 좋을 수밖에 없는 'SKT·KT·LGU+'…증권가 "매수해라"
LG그룹의 배터리 구하기..재무부담은 '가속화' [fn마켓워치]
2025-04-17
中 특수 사라지나…LG생건·호텔신라에 공매도 '우르르'
2025-04-18
"LG이노텍, 유리기판 사업 첫해…역성장 우려 해소"[클릭 e종목]
2025-04-19
2025-04-20
2025-04-21
LG화학 ELS 폭탄 터지나
2025-04-22
2025-04-23
‘머스크 테슬라 귀환’ 기대에 LG엔솔 주가도 강세 [종목Pick]
"LG디스플레이, 관세 우려에도 주가 상승 기대해볼 시기"-SK
2025-04-24
체질 개선 성공한 LG디스플레이…"연간 흑자도 기대"(종합2보)
"최대 매출에도" LG이노텍, 증권사 목표가 줄하향…왜
LG이노텍, 1분기 호실적에도 증권가 혹평에 급락[핫종목]
DS투자증권 "LG이노텍, 큰 반등 가능한 주가 수준"[아침밥]
"LG이노텍, 올해 감익 불가피…주가는 저평가"-흥국
2025-04-25
현대차증권 "LG디스플레이, 2분기 어려워 목표주가 하향"[아침밥]
2025-04-26
2025-04-27
2025-04-28
LG화학 담수사업부문, 글랜우드PE에 판다
세계2위 수처리사업 매각 LG화학 자금조달 숨통
LG전자, 1분기 매출 성장에도 주가는 우하향…"B2B·구독서비스 기대"
BNK證 “LG전자, 거시 환경 불확실성 높아… 목표주가 20% 하향”
2025-04-29
'최대 실적' LG CNS, 숨고르기…공모가 회복 언제
“드디어 박스권 탈출하나”…옛 ‘황제주’ LG생건 목표가 상향에 주가 ‘활짝’
LG생건, '체질 개선' 기대로 급등…증권가선 '갑론을박' [종목+]
‘1분기 실적 선방’ LG생건, 증권가 목표가 줄상향에 7% 강세 [줍줍리포트]
“해외 성장 기대” LG생활건강, 1분기 선방에 증권가 목표가 줄줄이 상향···주가도 ‘쑥’ [종목Pick]
"LG CNS, 예상 뛰어넘은 실적…클라우드 성장 가속화 구간"
"中선방한 LG생활건강, 미국선 아쉬워"[클릭 e종목]
LG화학 수처리 사업 1조원 대 매각… K조선, 컨테이너선 시장 ‘싹쓸이’[AI 프리즘*주식 투자자 뉴스]
2025-04-30
LG화학 1분기 흑자전환…고부가 매출 확대로 실적 개선(종합2보)
`역대 최대`도 지키지 못한 주가…한화에어로·삼전·LG엔솔, 실적발표 후 `뚝`
LG화학 1분기 흑자전환…석유화학 적자 축소, 고부가 매출 확대(종합)
흑자 전환했지만 예측도 어려운 수요…LG에너지솔루션 7% 급락
LG엔솔, 1분기 흑자 전환…불확실성에 시설투자 30% 줄인다(종합2보)
2025-05-01
2025-05-02
이웃집 불행은 나의 행복?…‘신규가입 중단’ SKT 주가 하락에 KT·LGU+ ‘신고가’ [투자360]
'신규가입 중단' SKT 1위 뺏길까…KT·LGU+ 신고가 '휘파람'
"저가매수 가능" LG에너지솔루션 투자의견 '매수'로 상향-DS
LG에너지솔루션, 유럽향 판매 둔화…"3Q부터 회복 예상"-NH
"LG화학, 2Q양극재 수요 둔화 불가피"[클릭 e종목]
NH투자증권 "LG엔솔, 유럽 판매 부진… 2분기 실적 우려"[아침밥]
LG화학, 내년부터 실적 대폭 개선 기대…목표가 12%↓-KB
"LG화학, 2분기 양극재 중심 실적 둔화…목표가↓"-하나
2025-05-03
'카브아웃 특화' 글랜우드PE, LG화학 담수사업 부문 품나[주간사모펀드]
2025-05-04
2025-05-05
2025-05-06
신고가 경신한 KT·LG유플러스... 증권가 "여름까지 우상향"
2025-05-07
SK텔레콤 대국민 사과에도 꺾인 투심…KT·LG유플러스 '신고가'
SK텔레콤, 유심 해킹 발표하고 9%대 '뚝'…KT·LG 반사이익
2025-05-08
LGU+ 1분기 영업익 15.6%↑…"보안·AI 경쟁력 강화할 것"(종합2보)
SKT 26만명 이탈…LG유플, 영업익 1조 재도전 [마켓딥다이브]
2025-05-09
'장 초반 반짝' 코스피, 2577.27로 하락 마감… LG엔솔은 2.9% 상승
LG유플러스, 업종 내 유일한 주주환원 모멘텀…목표가 상향-신한
2025-05-10
2025-05-11
과감한 체질개선에 LG유플 급등
2025-05-12
2025-05-13
2025-05-14
LG에너지솔루션, 단기 정책 불확실…조정시 매수 유효-키움
2025-05-15
인창개발 'LG전자 연구소' 개발사업, 착공 '시동'…다음달 철거 끝난다
2025-05-16
코스피, 0.21% 상승 마감… LG에너지솔루션 5.37% 하락
공모가도 깨진 LG엔솔…삼바에 시총 3위 내줘
브이원텍, 김민 전 LG디스플레이 모듈센터장 각자대표 선임
"LG엔솔 30만원도 붕괴"…'AMPC 리스크' 이차전지 개미들 비명
LG에너지솔루션, 공모가 30만원도 깨졌다…신저가 기록[핫종목]
"LG엔솔 추락 어디까지"…공모가 하회에 시총 4위로 밀려나[이런국장 저런주식]
LG엔솔 공모가 깨졌다…'생존싸움' 들어간 2차전지株 동반 약세
2025-05-17
2025-05-18
LG생건 화장품 '더후'…프리즈 뉴욕 참여 주목
2025-05-19
2025-05-20
'신용등급 올랐는데'…LG CNS 주주들 속터지는 이유
"바닥은 어디"…LG에너지솔루션, 사상 최저가 기록[핫종목]
코스피, 무덤덤 美 증시 따라 장 초반 강세 2620대 탈환 시도…LG엔솔은 또 최저가 [투자360]
"LG엔솔, LMR 배터리 수주는 매출 재평가 재료"-다올
2025-05-21
한화 세자릿수 치솟을 동안 LG는 뒷걸음질…그룹주 ETF 희비
'이것 없어지면 바로 적자전환' LG엔솔 역대 신저가로 내려앉은 이유는 [선한결의 이기업 왜이래]
2025-05-22
"LG전자, 구독 가전·WebOS 신성장 동력 될 것"[클릭e종목]
가전은 LG, 최선호주도 LG전자?…“전기전자株, 상승 탄력 강해질 것” [투자360]
2025-05-23
LG엔솔·에코프로 ‘신저가’…2차전지 ETF 수익률 꼴찌
2025-05-24
2025-05-25
OLED 혁신 통했다…LGD "올 실적 턴어라운드 원년" [스타즈 IR]
상장 후 최저가로 추락한 LG에너지솔루션[급등주 지금은]
2025-05-26
美 배터리 보조금 생존에 LG엔솔 모처럼 '활짝'…삼성SDI 4%↑[핫종목]
국내 증시, 장중 2600선 회복…LG엔솔·HMM `강세`
"車강판 지각변동" 트럼프, 日 US스틸 인수 승인…“올 실적 턴어라운드 원년” LGD, OLED 혁신 통했다 [AI 프리즘*주식 투자자 뉴스]
대신證 “LG유플러스, 통신 3사 중 가장 높은 무선 성장률…영업이익 1조원 복귀 전망″
2025-05-27
2025-05-28
코스피, 반도체·2차전지 강세에 2670선 마감…LG엔솔·POSCO홀딩스 5%↑
2025-05-29
2025-05-30
2025-05-31
2025-06-01
2025-06-02
2025-06-03
상반기 웃은 그룹주 ETF는 한화…LG는 울상
2025-06-04
2025-06-05
"덜 오른 지주사, 어디 있나요"…연기금은 LG·GS·삼성물산에 베팅 [이런국장 저런주식]
2025-06-06
“내가 제일 잘 나가”…올해 들어서만 24% ‘쑥’ LG유플러스 주가 신바람
2025-06-07
LG CNS, 한기평도 'AA' 상향…스플릿 해소[위클리 크레딧]
2025-06-08
2025-06-09
iM證 “LG엔솔, 7월 美 감세 법안 시행 시 실적 불확실성↑… 목표가 하향”
“LG에너지솔루션, 1분기 이후 실적 전망 불확실성”…목표가↓
"LG에너지솔루션, 1분기 이후의 실적 불확실성 확대…목표가↓"-iM
2025-06-10
2025-06-11
"'토종 공룡 OTT' 탄생하나" 티빙-웨이브 합병 조건부 승인…‘배터리 육성’ 의지 LG 구광모, 인니 공장 찾았다 [AI 프리즘*주식투자자 뉴스]
2025-06-12
LG CNS 효과에 '웃은' KB증권…IPO '왕좌' 유지
2025-06-13
LG CNS 날아올랐다…공모가 회복·거래량 급증
글랜우드PE, 1.4조원에 LG화학 워터솔루션사업부 인수 확정
LG이노텍, 환율 약세에 2Q 실적 부진 전망…주가는 저평가 구간-하나
2025-06-14
LG CNS "드디어 날았다" 상장 후 첫 공모가 회복 [이런국장 저런주식]
2025-06-15
2025-06-16
2025-06-17
LG CNS, 공모가 회복 후 6월에만 50% 올랐다···개미 ‘환호’ 왜? [종목Pick]
LG CNS, 연일 신고가 행진…사흘간 35% 급등[이런국장 저런주식]
2025-06-18
LG CNS, 연일 신고가 경신…공모가 돌파 후 주가 파죽지세
2025-06-19
'범LG家' 구본호, CSA 코스믹 등 상장사 지분 잇단 매입
“LG생건과 단 4칸 차이” 에이피알 1년만에 몸값 ‘5조’· 달바글로벌은 콜마 턱밑 추격 [투자360]
2025-06-20
2025-06-21
2025-06-22
2025-06-23
“이런 경사가”…새 정부 요직에 임원 들어갔다, 네이버·LG 주가 환호성
뷰티 3위 에이피알, LG생건 밀어냈다
배경훈 LG AI연구원장, 과기장관 후보 지목에…LG·LG전자 강세 전환
LG 출신 과기부장관에 로봇주 일제히 급등…로보티즈 '上'
"매력적인 밸류에이션"…LG전자, 순매수 1위 [주식 초고수는 지금]
2025-06-24
'장관 후보 배출' 깜짝 호재에 네이버·LG '급등'…다음 타자는 [분석+]
기업 출신 장관 인선…LG·NAVER 동반 '껑충'
2025-06-25
LG CNS 주식도 출퇴근길 거래 한다…대체거래소 첫 정기종목변경 [투자360]
빚투 6배 늘어난 LG씨엔에스… "적정주가 7만 원대" [마켓딥다이브]
2025-06-26
2025-06-27
"LG유플러스, 단기 셀온 부담…장기 주가 상승 기대"[클릭 e종목]
2025-06-28
2025-06-29
2025-06-30
상반기 주관 미래에셋·KB '양강' 형성…'LG CNS 효과'
K뷰티 화색 도는데…LG생활건강만 '찬바람'
"LG생건, 중국 시장 고전 중…현재가보다 주가 떨어질 것"-삼성
삼성증권 "LG생건, 중국인 떠나는데 미국인은 인디가 꽉"[아침밥]
"에이피알, LG생건 시총 추월 우연 아냐 …업종 내 우위 계속될 것"-유안타
LG생활건강, 인디 브랜드에 경쟁 밀려…목표가 10%↓-삼성
2025-07-01
LG, 지주회사 저평가 해소 기대감에 주가 상승세… MK시그널 추천 후 상승률 31.85% 기록[MK시그널]
2025-07-02
LG생활건강, 실적 아쉬움 이어져…보수적 관점 유지-한국
2025-07-03
“선두 굳히기”…‘LG CNS’ 앞세운 KB증권, 상반기 IPO 주관 1위
"테슬라발 훈풍에"…LG엔솔, 시총 3위 복귀[핫종목]
LG에너지솔루션, 장중 시총 3위 탈환…테슬라발 훈풍 영향 [이런국장 저런주식]
LG CNS ‘락업 해제’ 한 달 앞…FI 수익 정점 근접
테슬라 급등에 2차전지株도 '반짝'…LG엔솔 1%대 강세[핫종목]
LG CNS 락업 D-1개월, FI는 표정관리 중 [종목Pick]
2025-07-04
LG화학 기초소재 사업부 판다…1.5조원 이상 거론 [시그널]
LG에너지솔루션 시총 3위 재탈환
한계 다다른 LG화학, BPA 사업부 매각한다
코스피, 0.91% 내린 3080선… LG에너지솔루션 2%대 하락세
이차전지 맏형 LG엔솔, 시총 3위 ‘탈환’…테슬라 로보택시 타고 반등 성공할까 [종목Pick]
2025-07-05
2025-07-06
LG화학·롯데케미칼 등 석유화학株 깜짝 동반 상승…왜?
2025-07-07
2분기 실적 시즌 개막에 '희비'…LG전자↓·LG엔솔↑[핫종목]
LG전자, '어닝 쇼크' 여파…3%대 하락 [종목+]
코스피, 장중 보합권서 등락…'깜짝 실적' LG엔솔 3.5%↑
LG엔솔, 2분기 '깜짝 실적' 내놨지만…급등 후 약세 전환
‘관세불안’에 코스피 긴장…‘2분기 흑자’ LG엔솔, 2%대 강세
LG, 상법개정으로 밸류업 기대…목표가는 유지-iM증권
"LG이노텍, 2분기 실적 기대치 밑돌 것…목표가↓"-상상인
2025-07-08
"EV 보조금 폐지 앞서 배터리 매입↑"…LG엔솔 '깜짝 실적' 배경은
삼성證 "LG화학, LG엔솔 지분 활용 가능성…목표가↑"[줍줍 리포트]
LG디스플레이, 연간 흑전 기대…하반기 실적 기대감 유효-SK
2025-07-09
어닝서프라이즈에도…LG에너지솔루션, 시총 3위 쉽지 않은 이유
LG전자, 목표가 줄하향…“TV 수요 둔화·관세 리스크 주요인”[투자360]
2025-07-10
KT·LG유플 목표가 줄상향…‘해킹 사태’ SKT, 지금이 매수 타이밍? [투자360]
코스피, 강보합 출발 후 또 연고점 돌파…삼바·LG엔솔 2%대 강세
2025-07-11
2025-07-12
2025-07-13
6분의 1 토막난 '황제주' LG생활건강[급등주 지금은]
2025-07-14
"HBM 더 얇게 만든다" LG, ‘꿈의 장비’ 도전…'주가 3배' 뛴 에이피알, 1300억 중간 배당 [AI 프리즘*주식투자자 뉴스]
LG유플러스, 펀터멘털 개선 뚜렷…목표가↑-유진
2분기 배터리 실적 엇갈릴 전망…LG엔솔 깜짝실적 다음은? [투자360]
2025-07-15
2025-07-16
2025-07-17
2025-07-18
한 달 만에 30% 뛴 LG화학, 순매수 1위 [주식 초고수는 지금]
2025-07-19
2025-07-20
LG생건 앞지른 에이피알, 공모가 대비 3배 올라[급등주 지금은]
2025-07-21
"원·달러 환율 치솟아도 산다"…外人 찜한 키움證·LG유플
"LG화학 바닥 지났다"…하반기 실적 개선 기대감에 6% 강세[줍줍 리포트]
"LG화학, 양극재·석유화학 모두 바닥은 지났다…목표가↑"-키움
2025-07-22
LG유플, '1000억 깜짝 자사주 소각'…장중 52주 신고가[이런국장 저런주식]
11년 만에 단통법 폐지…LG유플 '1조클럽' 시동 [마켓딥다이브]
2025-07-23
2025-07-24
되살아난 이차전지株…LG엔솔, 특허소송 승소에 10% 강세[핫종목]
LG엔솔, ‘배터리 특허침해’ 소송 승소에 9% 급등 마감 [종목Pick]
영업익 전분기 대비 13배?…LG이노텍 3분기 반등 시나리오 [종목Pick]
LG에너지솔루션, 배터리 특허침해 소송 승소에 8%대 급등
코스피, 외국인 매수에 강세…LG엔솔 8%대 '급등'
유안타證 “LG씨엔에스, 스테이블코인 수혜 가능… 목표가↑”
"LG CNS, AI·클라우드에 스테이블코인까지…겹호재에 목표가 ↑"-유안타
LG씨엔에스, 매출 회복·스테이블코인 수혜 기대…목표가↑-유안타
2025-07-25
"거래량 12배 터졌다" LGD, 실적 반등 기대에 9% 상승[핫종목]
LG엔솔 2분기 영업익 4천922억원…美 ESS 확대 승부수(종합2보)
LG CNS 락업물량 내달 해제…주가 변곡점 되나
'2분기 적자' LG디스플레이, 하반기 기대감에 8% '상승'
2025-07-26
2025-07-27
LG유플러스, 자사주 1000억 소각…밸류업 보폭 키운다[스타즈IR]
“LG유플러스 주가 15% 오를 것…최고 2만 원 기대” [스타즈IR]
하나證 “LG엔솔, 2년 만에 이익 추정치 상향… 투자의견 ‘중립→매수’”
2025-07-28
하나증권 "LG엔솔, ESS 생산으로 이익 증가 기대"[아침밥]
“LG에너지솔루션, 2년만의 이익 추정치 상향 국면”…투자의견·목표가↑
"LG엔솔, 낙관과 비관 공존…기업가치 상향 돌파구 필요"-메리츠
"LG엔솔, 반등 지속보단 단기 반등에 가까워"[클릭 e종목]
LG에너지솔루션, ESS 고성장에 단기 반등 기대…목표가↑-NH
LG에너지솔루션, "2년만 이익 추정치 개선"…목표가 45만-하나
"LG전자, 3분기 美 관세 영향 본격화…목표가↓"-하나
2025-07-29
2025-07-30
2025-07-31
LG CNS, 내달 5일 FI 지분 21% 락업 풀린다
삼성 이어 LG엔솔도 상승···투자자들 “테슬라는 한국 기업”
"LG에너지솔루션, 구조적 상승기 진입…목표가↑"-다올
“LG에너지솔루션, 대규모 ESS 배터리 수주에 구조적 상승기 진입”…목표가↑
2025-08-01
2025-08-02
'달라진 시선'…증권사 10곳, LG엔솔 목표가 높여[청기홍기]
2025-08-03
LG엔솔 '6조' 잭팟에 확 달라진 분위기…2차전지주 다시 불붙나
2025-08-04
LG씨엔에스, 67% 보호예수 해제?FI·최대주주 물량 동시 출회
2025-08-05
맥쿼리PE, LG CNS 지분 5.6% 장외 매각
'자사주 소각' LG유플 날개 외인 지분도 SKT 역전눈앞
‘자사주 모멘텀’ LG유플 상승세 지속…SK텔레콤 외국인 지분율 역전도 가시권
‘AI 훈풍’ LG CNS, 오버행 떨치고 주가 탄력받나
"AI 전환 수혜"…LG씨엔에스 21.5% 오버행 우려 불식 [마켓딥다이브]
'아픈 손가락' LG디플 흑자 반전?…"드디어 배당" LG전자 주주 설렌다
LG CNS, 소버린AI·스테이블코인 간접수혜…목표가 9만 -신영
2025-08-06
맥쿼리PE, LG CNS 보유 지분 5.6% 블록딜로 엑시트 [fn마켓워치]
LG유플이 SKT 따라 잡는다고?...외국인 지분율 역전 눈앞
2025-08-07
VIG파트너스, LG화학 에스테틱 사업 품는다…2000억 베팅[투자360]
LG화학, 에스테틱 사업부 VIG파트너스에 매각
"눈높이 낮춘 덕에..." LG화학, 에스테틱 매각 성공 [fn마켓워치]
이춘석 의원은 LG CNS 주식을 언제 샀을까? [이런국장 저런주식]
2025-08-08
야구 순위도 1·2위 주식 인기도 1·2위 한화·LG 둘다 사볼까
2025-08-09
VIG파트너스, 비올 이어 LG화학 사업부 품었다…에스테틱 강화[주간사모펀드]
2025-08-10
2025-08-11
에스테틱 매각한 LG화학…LG에너지솔루션 지분 향방은[마켓인]
LS證 “LG화학, LG엔솔 지분 대신 에스테틱 매각… ‘중립’ 의견으로 하향”
LG유플러스, 실적 턴어라운드에 주주환원도 확대 전망…유진
2025-08-12
2025-08-13
中 OLED 미국 퇴출 소식에 LG디스플레이 20% ‘불기둥’ [종목Pick]
미국서 中 OLED 퇴출 소식에…LG디스플레이 19%대 '급등'
中 OLED 미국서 15년간 퇴출…LG디플 ‘반사이익’ 기대에 상승 [줍줍리포트]
2025-08-14
코스피, 0.04% 상승한 3225.66 마감… 'LG엔솔' 1%대 올라
2025-08-15
2025-08-16
LG에너지솔루션·테스·셀비온[하나證 주간추천주]
2025-08-17
2025-08-18
2025-08-19
LG전자 손자회사 알폰소 "1조弗 커넥티드TV 시장 선도…나스닥 상장 추진"
2025-08-20
2025-08-21
삼성전자 지분설에도 "9만 갑니다" 최선호주… ‘북미 의존도 부담’ LG엔솔, 목표가 32만원 상향 [株토피아]
2025-08-22
"석유화학 재편, 연말 구체화...톱픽 LG화학·롯데정밀화학"
2025-08-23
2025-08-24
2025-08-25
개인 매도에도 코스피 3200선 회복… SK하닉·LG엔솔 등 3%대↑
2025-08-26
코스피, 미 연준 이사 해임에 낙폭 확대… SK하닉·LG엔솔 하락 전환
LG이노텍, 바닥 이후 반등…하반기 반전 기대-유안타
2025-08-27
50% 오른 LG유플, 뒷걸음질 SKT…"하반기엔 바뀐다"
2025-08-28
LG, 2500억원 규모 자사주 소각…5%대 강세[핫종목]
㈜LG, 2천500억원 규모 자사주 소각…첫 중간배당도 실시
LG, 2500억원 자사주 소각 결정에 5%대 급등
한투증권 "코스피 대형주 지수에 LG CNS 등 8종목 편입 예상"
2025-08-29
2025-08-30
2025-08-31
해킹사태 역대급 과징금에 또 갈렸다…개미는 SKT ‘줍줍’ vs 외국인 LG유플 ‘사자’
‘역대급 과징금’에도 개미 픽은 SK텔레콤…외국인 사들이는 LG유플러스와 승패는
2025-09-01
2025-09-02
2025-09-03
대신證 “LG유플, 해킹 이슈 약세 불가피… 우려 해소 시 반등”
2025-09-04
"2차전지주 선별 접근해야"...LG엔솔·나노신소재 기대
코스피 3200선 회복…LG엔솔, 벤츠 공급 계약 소식에 2차전지株 강세 [투자360]
코스피, 오늘도 ‘눈치보기’…LG에너지솔루션 3%↑
2025-09-05
미 당국, 현대차·LG엔솔 합작 배터리 공장 압수수색…"한국인 300명 등 약 450명 체포"
2025-09-06
2025-09-07
2025-09-08
곰팡이 수용소, 수백명 구금…LG에너지솔루션 투자심리는 선방
LG엔솔·현대차 주가, 미국發 변수에 흔들…실적 전망 ‘먹구름’
미국서 한국인 직원 구금 사태에…"LG엔솔, 내년 판매량 타격 불가피"
美 HMGMA 중단 직격탄…현대차·LG엔솔 동반 하락
코스피, 외인·기관 '사자'에 3200선 강보합…현대차·기아·LG엔솔 '약세'
美, 한국인 근로자 구금에 공장 타격…LG엔솔 주가 흔들 [선한결의 이기업 왜이래]
美 배터리공장 단속 여파… LG엔솔 판매 차질·건설사 인력난 우려
코스피, 3200선 강보합 출발…‘구금 사태’ LG엔솔·현대차↓
코스피, 외국인·기관 매수에 강보합…현대차·기아·LG엔솔 '약세'
코스피, 0.15% 상승 강보합 출발…‘美 공장 비상’ LG엔솔·현대차↓
LS증권 "LG엔솔-현대차 합작공장 공사 중단 타격 우려"[아침밥]
LG에너지솔루션, 유럽·북미향 수주로 중장기 성장 모멘텀…'매수'-신한
"LG에너지솔루션, 어려운 환경 속 빛난 수주 경쟁력"-신한
2025-09-09
美 조지아 구금 여파에도…“현대차·LG엔솔, 실적·주가 충격 제한적” [투자360]
유안타證 "LG화학, 글로벌 석화 폐쇄 사이클 수혜 기대"
"LG화학, 글로벌 석화 폐쇄 사이클 수혜…목표가 유지"[클릭 e종목]
LG화학, 글로벌 석화 폐쇄 사이클 수혜 기대…'매수'-유안타
2025-09-10
“LG화학, 문제는 4분기부터…주가 15% 더 떨어질 듯”
LS證 “LG화학, 4분기 이후가 문제… 목표가 내리고 ‘중립’ 유지”
2025-09-11
폭스콘 '웃고' LG이노텍 '울고'…"다 같은 애플 수혜주? NO!"
AI가 부른 지각변동... 부품사 삼성전기 시총이 가전업체 LG전자 추월
신영증권 "LG화학 내년부터 긍정 사이클 도래할 것"[아침밥]
2025-09-12
2025-09-13
산업 구조조정에 주주환원 기대까지…LG화학 살아날까 [한경우의 케이스스터디]
2025-09-14
2025-09-15
임우형 LG AI연구원장 "LG화학 공장, AI 적용해 100억 이득" [KIW 2025]
"지주사 중 상승폭 제한된 LG…추가 모멘텀 땐 긍정적"-BNK
2025-09-16
'4위' LG그룹 추격자 누구길래…올해만 '시총 55조' 뛰었다
뭉치는 HD현대, 그룹 시총 134조…한화 제치고 LG 추격 [마켓딥다이브]
"바닥쳤다" LG디스플레이, 턴어라운드 기대... SK하이닉스, 디램·낸드 업황 견조 [株토피아]
“하반기 턴어라운드 기대” LG디스플레이 주가 9% 급등 [종목Pick]
"LG디스플레이, 하반기 대규모 턴어라운드 기대…목표가↑"[클릭 e종목]
2025-09-17
2025-09-18
넥스트레이드, LG씨엔에스 등 66개 종목 추가 거래 중단
대체거래소에서 LG씨엔에스도 거래 못 한다…66개 종목 추가 중단
희망퇴직으로 현금 흐름 개선…LG전자·LG전자우 5% 급등 [줍줍 리포트]
불장서 소외된 LG엔솔…목표가도 줄하향
2025-09-19
2025-09-20
LG이노텍·대한유화·신세계[하나證 주간추천주]
2025-09-21
2025-09-22
LG전자, B2B 주목…외국인 이달 1200억 순매수
iM證 “LG이노텍, 5년 만에 증익 가능성 커져… 목표가 상향”
이달 들어서만 1200억 샀다…LG전자, 5개월여 만에 외인 비중 30% ‘탈환’ [종목Pick]
LG이노텍, 5년 만에 증익 기대감… 목표가 23만원 ↑-iM
2025-09-23
아이폰17 흥행 기대감…LG이노텍 등 부품주 '들썩'
2025-09-24
2025-09-25
2025-09-26
LG생활건강, 3분기 실적 먹구름에 신저가 기록[이런국장 저런주식]
2025-09-27
2025-09-28
LG씨앤에스, 파라다이스, 아세아… 삼성證이 꼽은 코스피200 신규 편입 예상 종목
2025-09-29
'국가전산망 화재 시작점' LG엔솔 배터리…주가에 영향은? [선한결의 이기업 왜이래]
2025-09-30
"LG디스플레이 4년 만에 최대 실적"…하이브, BTS 모멘텀에 목표가 38만원↑ [株토피아]"
LG전자, 인도법인 상장 이르면 내달 마무리…1.8조원 확보 예상(종합)
2025-10-01
LG전자, 印·美 '더블 상장'…1.8조 실탄 미래사업 베팅 [시그널]
NH증권 "LG전자, TV 부진에 3분기 실적 예상 밑돌 것"[아침밥]
“예상보다 좋은 아이폰 성적, 옅어진 LG이노텍 먹구름”
제LG디스플레이, 3분기 실적 시장 기대치 상회 전망…목표가 25% ↑-IBK
"LG디스플레이, 미국 OLED 공급망의 유일한 대안…목표가↑"-IBK
2025-10-02
시간외매매서 13% 급등한 LG엔솔···왜?
LG에너지솔루션, 장 막판 14% 급등 [이런국장 저런주식]
'재무 부담' 커진 LG화학, LG엔솔 지분 2조 매각 [마켓딥다이브]
2025-10-03
2025-10-04
2025-10-05
2025-10-06
2025-10-07
탑런토탈솔루션, LG디스플레이 검사장비 첫 수주…밸류체인 확장 시동
2025-10-08
2025-10-09
김영재 LG전자 상무 "휴머노이드, 가사노동 해방이 목표"[주톡피아]
한화그룹 시총 1년 새 3배 성장했다…LG·포스코는 뒷걸음질[이런국장 저런주식]
2025-10-10
코스피 천장 뚫었는데…LG엔솔 10% 폭락
2025-10-11
2025-10-12
LG전자 '사라' vs LG생건 '글쎄'
"가전·TV는 부진, 전장은 방어"…삼성·LG전자 3분기 실적 향방
2025-10-13
"LG이노텍, 아이폰17 힘입어 실적 개선…목표가↑"-유안타
2025-10-14
LGD·에코프로비엠·호텔신라…3분기 실적 개선 기대株 뜬다
LG전자 인도법인 현지상장…첫날 주가 50% 급등, 본사 시총 넘어(종합)
초고수, 이차전지 호재 맞은 LG에너지솔루션 매수 [주식 초고수는 지금]
LG에너지솔루션 어닝 서프라이즈! 2차전지株, 단기 반등 vs 추세 회복? [투자 일기예보]
LG전자, 인도증시 상장 첫날 50% 폭등
LG전자는 연중 ‘마이너스’인데… 인도법인, 상장 첫날 50% 상승
간밤 테슬라 급등하더라니…LG에너지솔루션·화학 등 ‘이차전지주’ 상승세
LG에너지솔루션, 내년 상반기까지 호실적…장중 6% 급등 [줍줍 리포트]
'2분기 연속 흑자' LG엔솔…증권가 전망은 엇갈려 [종목+]
"LG에너지솔루션, 3Q 실적 무난하지만 낙관 힘들어"[클릭 e종목]
"LG유플러스, 내년 DPS 상승 가능성 보고 장기 매수"[클릭 e종목]
2025-10-15
인도 증시 입성한 LG전자…“신용등급 상승에 긍정적” 호평 [종목Pick]
인도법인 상장·실적 호조…LG전자, 저평가 털고 반등할까
LG엔솔, '연휴 폭락' 후 반등 성공…증권가는 엇갈린 전망
2차전지株 약세…LG엔솔·LG화학 1%대 동반 하락 [투자360]
"LG이노텍, 5년만에 실적 개선 국면 진입…목표가↑"[클릭 e종목]
2025-10-16
LG엔솔 8.8%·엘앤에프 19.27%…2차전지 관련주 급등
"LG화학, 엔솔 빼면 실적 부진"... HD현대마린솔루션, 내년까지 호실적 전망 [株토피아]
2025-10-17
LX홀딩스, LG광화문빌딩 5120억원에 인수
2025-10-18
2025-10-19
'GSK·LG화학 파트너' 쿼드메디슨 목표시총 1700억 근거는
2025-10-20
"美中 무역 갈등 수혜" LG그룹 시총 3위로[이런국장 저런주식]
미·중 갈등에 반사이익 LG…그룹시총 3위
LG전자 급등 미리 알았다…투자의 질 높이는 에픽AI
인구 14억 인도서도 ‘가전은 LG’…印 IPO 성공에 주가·목표가 ‘쑥’ [종목Pick]
"인도 IPO 성공" LG전자 목표가 ↑... 한미약품, 신약 개발 기대감에 업종 내 '최선호주' [株토피아]
"인도법인 PBR 20배인데 본사 0.7배"…LG전자 4%대 강세
인도 증시 연착륙한 LG전자, 성장 동력 확보로 목표가 ↑ [줍줍 리포트]
2025-10-21
'나도 이제 9만 전자'…숨통 트이는 LG그룹, 시총 3위 재탈환
애플 신제품 효과?…‘부품 공급’ LG이노텍 52주 신고가·비에이치 약세 [투자360]
'HBM 1황' SK하이닉스, 기술·제품 모두 시장 우위… LG전자 '어닝 서프라이즈' [株토피아]
"아이폰17 대박 났다"…LG이노텍 23만원 돌파 '신고가' [종목+]
한 달 만에 20% 뛰었다…'애플 효과'에 반등한 LG이노텍 [종목+]
2025-10-22
코스피 불장에 돌아온 행동주의펀드…“LG화학, 주주이익 높여라”
[단독] "LG화학, 엔솔지분 매각해 자사주 매입을"
英 행동주의 펀드, LG화학 공습…"이사회 개편하라" [시그널]
"저평가 심각" 행동주의 펀드의 경고…LG화학 장중 '52주 신고가'
“이 주가는 말이 안 된다”…장 초반 12% 급등한 LG화학, 왜?
英 헤지펀드 “LG화학 저평가 됐다” 주장에 장중 ‘불기둥’  [종목Pick]
LG화학, 英 행동주의 "주가 저평가"에 10% 넘게 급등[이런국장 저런주식]
美中 무역갈등 반사이익 본 LG그룹, 시총 3위 자리 되찾다
2025-10-23
[단독]개미만 모르는 위험신호...국민연금, LG화학에 비공개 경고
"저평가" 한마디에… LG화학, 13% 급등
2025-10-24
"LG, 배당 안정성과 배당성향 주목…목표가↑"[클릭 e종목]
2025-10-25
LG화학 향한 행동주의 공세, 해프닝으로 끝나나[위클리IB]
2025-10-26
2025-10-27
2025-10-28
2025-10-29
2025-10-30
전기차 캐즘에도 LG엔솔 3분기 선방…ESS 생산 극대화 추진(종합2보)
2025-10-31
"LG에너지솔루션, ESS 전망 밝지만 EV 판매 급감…투자의견 '홀드'로"-삼성증권
2025-11-01
2025-11-02
'슈퍼위크' 옥석 가리기…두산에너빌 웃고 SKT·LG엔솔 울고
2025-11-03
개미 3000억원 몰렸는데...옛 황제주 LG생활건강 향방은?
LG화학, 업황개선·LG엔솔 지분활용 주목…목표가 35%↑
LG화학, 4분기 영업익 대폭 감익 전망…투자의견 '중립'-하나
"LG화학, 내년 실적개선·LG엔솔 지분활용 기대"[클릭 e종목]
LG전자, 1분기부터 반등 기대…목표가 16%↑-iM
LG화학, 실적 개선ㆍLGES 지분 활용 기대...목표주가 50만원↑-iM
2025-11-04
'빅3' 마지막 퍼즐은… LG vs 현대차 200조 클럽 문턱서 격돌
"아, 옛날이여" LG생건  '황제주 영광' 되찾을까
2025-11-05
[단독]맥쿼리운용, LG CNS 지분 7% 블록딜 [시그널]
2025-11-06
"LG유플러스, 견조한 주가 흐름 예상…목표가 유지"[클릭 e종목]
2025-11-07
해킹 여파에 흔들린 'SKT·KT'… 하반기 LG유플러스 주가 7% ↑
2025-11-08
2025-11-09
LG디스플레이, OLED 중심 체질개선…4년만의 흑자 보인다
"너무 떨어졌나?" 관심받는 LG그룹주
2025-11-10
차익실현 나선 외국인, SK스퀘어·LG그룹주는 샀다
SK증권 "LG전자, AI없이는 주가 반등 쉽지 않다"
SK증권 "LG전자 AI 미래전략 부족, 주가는 30% 뛴다"[아침밥]
2025-11-11
2025-11-12
2025-11-13
2025-11-14
LG, 계열사 가치상승·배당 분리과세 가능성…목표가↑-NH
"LG화학, 미국 기업에 3.8조 양극재 공급…'가뭄에 단비 같은 소식'"-메리츠
"LG화학, 양극재 공급계약 체결…긍정적인 고객사 다변화"[클릭 e종목]
"""

# 2. 데이터 파싱 로직
data_list = []
current_date = None
date_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}') # 날짜 형식(YYYY-MM-DD) 감지

# 텍스트를 줄 단위로 나눕니다.
lines = raw_text.strip().split('\n')

for line in lines:
    line = line.strip()
    if not line: continue # 빈 줄 건너뜀

    # (1) 날짜인지 확인
    if date_pattern.match(line):
        current_date = line # 날짜 갱신 (이제부터 나오는 제목은 이 날짜 것)

    # (2) 날짜가 아니면 '제목'으로 저장
    else:
        if current_date: # 날짜가 먼저 나와서 세팅된 상태라면
            data_list.append({
                'date': current_date,
                'title': line
            })

# 3. 데이터프레임 생성
df = pd.DataFrame(data_list)

# 4. 결과 확인 및 저장
print(f"총 {len(df)}개의 뉴스가 추출되었습니다.")
print("-" * 30)
print(df.head(10)) # 상위 10개 미리보기

# CSV 파일로 저장 (한글 깨짐 방지 utf-8-sig)
df.to_csv("/content/drive/MyDrive/Colab Notebooks/2025-2/[25.9~25.12] Bamboo DnA/DnA_project (Stock_and_News)/Data/LG_energysolution/News/naver_finance_titles_LG_total_2025.csv", index=False, encoding="utf-8-sig")

총 600개의 뉴스가 추출되었습니다.
------------------------------
         date                                 title
0  2025-01-01      1월 IPO 기업 9곳…'6조 몸값' LG CNS 출격대기
1  2025-01-02          공모주 한파 이겨낼까 … LG CNS 등 9곳 출격
2  2025-01-02       '6조 대어' LG CNS, 얼어붙은 공모주 시장 녹일까
3  2025-01-02    LG이노텍, 4분기 실적 부진 전망에 새해 첫 거래일부터 하락
4  2025-01-02     메리츠證 "LG이노텍, 고객사 내 경쟁 심화 우려…목표가↓"
5  2025-01-02  LG이노텍, 4Q 영업익 시장 예상치 하회 전망…목표가↓ -메리츠
6  2025-01-03          삼성SDS 주가 하락에…상장 앞둔 LG CNS 고심
7  2025-01-05        혹한기 겪은 IPO 시장, LG CNS로 `봄` 맞을까
8  2025-01-05     업력 39년 'LG CNS'…왜 지금 IPO할까[공모주달력]
9  2025-01-06                LS증권, 'LG·우투증권 DNA' 강화


In [ ]:
import pandas as pd

input_filename = "/content/drive/MyDrive/Colab Notebooks/2025-2/[25.9~25.12] Bamboo DnA/DnA_project (Stock_and_News)/Data/LG_energysolution/News/naver_finance_titles_LG_total_2025.csv"

try:
    df = pd.read_csv(input_filename)

    # -------------------------------------------------------
    # 2. 찾고 싶은 두 단어 설정
    word1 = "LG에너지솔루션"
    word2 = "LG엔솔"
    # -------------------------------------------------------

    # 3. 필터링 (AND 조건: & 사용)
    # 두 조건식 사이에 '&' 기호를 넣으면 됩니다.
    mask = df['title'].str.contains(word1, na=False) | df['title'].str.contains(word2, na=False)

    filtered_df = df[mask]

    # 4. 결과 확인
    print(f"'{word1}'와(과) '{word2}'가 모두 포함된 뉴스를 {len(filtered_df)}건 찾았습니다.\n")
    print(filtered_df)

    # 5. 저장
    output_filename = f"/content/drive/MyDrive/Colab Notebooks/2025-2/[25.9~25.12] Bamboo DnA/DnA_project (Stock_and_News)/Data/LG_energysolution/News/naver_finance_titles_LG에너지솔루션_LG엔솔_2025.csv"
    filtered_df.to_csv(output_filename, index=False, encoding="utf-8-sig")

except FileNotFoundError:
    print("CSV 파일이 없습니다.")

'LG에너지솔루션'와(과) 'LG엔솔'가 모두 포함된 뉴스를 155건 찾았습니다.

           date                                            title
18   2025-01-09             길어지는 '이차전기 혹한기'…LG에너지솔루션 4% '뚝'[핫종목]
21   2025-01-09                   LG에너지솔루션 적자 전환 '실적 쇼크'에 4%대 급락
28   2025-01-10          NICE신평, LG화학 등급전망 ‘부정적’…LG엔솔 실적 연동[마켓인]
31   2025-01-10            LG엔솔 '어닝쇼크'…"일회성 비용 여파, 테슬라 신차 수혜 기대"
35   2025-01-11                업황 악화에 LG엔솔도 부진…위기의 LG화학[위클리 크레딧]
..          ...                                              ...
578  2025-10-30         전기차 캐즘에도 LG엔솔 3분기 선방…ESS 생산 극대화 추진(종합2보)
579  2025-10-31  "LG에너지솔루션, ESS 전망 밝지만 EV 판매 급감…투자의견 '홀드'로"-삼성증권
580  2025-11-02               '슈퍼위크' 옥석 가리기…두산에너빌 웃고 SKT·LG엔솔 울고
582  2025-11-03                 LG화학, 업황개선·LG엔솔 지분활용 주목…목표가 35%↑
584  2025-11-03             "LG화학, 내년 실적개선·LG엔솔 지분활용 기대"[클릭 e종목]

[155 rows x 2 columns]
